In [1]:
%load_ext autoreload 
%autoreload 2
    
from elasticdino.training.train_elasticdino import train
from IPython.display import display

In [1]:
from elasticdino.training.ablations import AblateDeformations, AblationTaskHeads, HypersimTaskHeads
from elasticdino.training.util import debug_features
from elasticdino.model.elasticdino import ElasticDino
from elasticdino.training.losses import reproduction_loss
import torch
from accelerate import Accelerator
from accelerate.utils import set_seed, DistributedDataParallelKwargs
from PIL import Image

import torch
import torchvision
from accelerate import Accelerator
from accelerate.utils import set_seed, DistributedDataParallelKwargs
import numpy as np

def ablation_loss(results_list, batch, features, min_scale, use_blur):
  rpr = 0
  diffl = 0
  diffr = 0
  resl = 0
  norml = 0
  bll = 0
  n = 0

  if use_blur:
      features = torch.nn.functional.interpolate(features, scale_factor=0.5, mode="bilinear")

  for r, f in results_list:
    resolution = batch["images"].shape[-1]
    scale = 1.0
    while resolution >= min_scale:
      n += scale
      # Check if key exists in dictionary before attempting to access it
      rpr += scale * reproduction_loss(r.get("reproduced"), batch["images"], resolution, order=0) if "reproduced" in r else 0
        
      diffl += scale * reproduction_loss(r.get("diffuse_illuminations"), batch["diffuse_illuminations"], resolution, order=2) if "diffuse_illuminations" in r else 0
        
      diffr += scale * reproduction_loss(r.get("diffuse_reflectances"), batch["diffuse_reflectances"], resolution, order=2) if "diffuse_reflectances" in r else 0
        
      resl += scale * reproduction_loss(r.get("residuals"), batch["residuals"], resolution, order=2) if "residuals" in r else 0
        
      norml += scale * reproduction_loss(r.get("normal_bump_cams"), batch["normal_bump_cams"], resolution, order=2) if "normal_bump_cams" in r else 0
      resolution //= 2
      scale *= 2
        
    if use_blur:
      downscaled = torch.nn.functional.interpolate(f, features.shape[-1], mode="bilinear") 
      bll += (downscaled - features).abs().mean()
      
    del r

  rpr = rpr / n
  diffl = diffl / n
  diffr = diffr / n
  resl = resl / n
  norml = norml / n
  bll = bll / len(results_list)


  return rpr + diffl + diffr + resl + norml + bll 
  
  
def debug_step(batch, features, results, running_loss, n):
  line = f"{n} {running_loss}"
  print(line)
  res = debug_features(features[0], [results[-1][0]])
  img = torchvision.transforms.functional.to_pil_image(batch["images"][0]).resize((128, 128))
  res = Image.fromarray(np.hstack([img, res]))
  display(res)


def train(train_config,
          model_config,
          get_models,
          get_dataloader):
  
  set_seed(42)
  dynamo_backend = "no" # "inductor"
  accelerator = Accelerator(mixed_precision="fp16", dynamo_backend=dynamo_backend)
  lr = train_config.get("lr", 1e-4)
  max_iterations = train_config.get("max_iterations", None)
  debug_interval = train_config.get("debug_interval", 50)
  save_interval = train_config.get("save_interval", 1000)
  use_blur_loss = train_config["use_blur_loss"]
  start_size = model_config["start_size"]
  upscaler, task_heads = get_models()
  optimizer = torch.optim.AdamW(
      [{"params": upscaler.parameters(), "lr": lr},
      {"params": task_heads.parameters(), "lr": lr}], eps=1e-5, weight_decay=0.0)

  dataloader = get_dataloader()
  dataloader, upscaler, task_heads, optimizer = accelerator.prepare(dataloader, upscaler, task_heads, optimizer)

  running_loss = None

  n = 0
  try:
    for epoch in range(5):
        print("Epoch", epoch + 1)
        for images, diffuse_illuminations, diffuse_reflectances, residuals, normal_bump_cams in dataloader:
            batch = dict(
                images=images,
                diffuse_illuminations=diffuse_illuminations,
                diffuse_reflectances=diffuse_reflectances,
                residuals=residuals,
                normal_bump_cams=normal_bump_cams,
            )
            if n == max_iterations:
              return
            n += 1

            with accelerator.autocast():
              results, features = upscaler(batch["images"], return_all_scales=True, return_original_features=True)
              loss = ablation_loss([(task_heads(r), r) for r in results], batch, features, start_size, use_blur_loss)
                          
            accelerator.backward(loss)
            optimizer.step()
            optimizer.zero_grad()

            if n % debug_interval == 0 and accelerator.is_local_main_process:
              debug_step(batch, features, results, running_loss, n)

            if running_loss is None:
              running_loss = loss.item()
            else:
              running_loss = 0.99 * running_loss + 0.01 * loss.item()

            del batch
            del loss
            del results
            del images
            del features
            del normal_bump_cams
            del residuals
            del diffuse_illuminations
            del diffuse_reflectances
            

  except:
    del batch
    del loss
    del optimizer
    del upscaler
    del results
    raise
   
model_config = dict(
        dino_model="l",
        n_features_in=1024,
        layers={
            32: dict(hidden_features=512, n_blocks=5, layers_per_block=8),
            64: dict(hidden_features=256, n_blocks=4, layers_per_block=8),
            128: dict(hidden_features=256, n_blocks=3, layers_per_block=8),
        },
        start_size=32,
        target_size=128,
)

def get_models(ablation):
    def get_ablated():
        if ablation == "heads":
            return ElasticDino(model_config), AblationTaskHeads(model_config["n_features_in"])
        elif ablation == "deformations":
            return AblateDeformations(model_config), HypersimTaskHeads(model_config["n_features_in"])
        elif ablation == "none":
            return ElasticDino(model_config), HypersimTaskHeads(model_config["n_features_in"])
        else:
            raise Exception("Unknown ablation")
    return get_ablated

from elasticdino.data.hypersim import HypersimDataset

hypersim_path = "/mnt/home/mizrahiulysse/datasets/hypersim/"

ablation = "deformations"

train_config = dict(
  n_epochs=1,
  max_iterations=2,
  lr = 1e-4,
  debug_interval=1,
  save_interval=5,
  batch_size=4,
  use_blur_loss=ablation == "deformations"
)


def get_dataloader():
    dataset = HypersimDataset(hypersim_path)
    return torch.utils.data.DataLoader(dataset, batch_size=train_config["batch_size"], shuffle=True, num_workers=16)

args = (train_config, model_config, get_models(ablation), get_dataloader)

from accelerate import notebook_launcher

notebook_launcher(
  train,
  args,
  num_processes=2,
)


KeyError: 'ABLATION'

In [ ]:
"